In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# Author: Dave Wang

## Get started

### Install required packages

First, we'll install the necessary packages.


In [ ]:
# %pip install --upgrade --quiet "a2a-sdk>=0.3.4" --force-reinstall --quiet
# %pip install --upgrade --quiet "google-cloud-aiplatform[agent_engines, adk]>=1.112.0" --force-reinstall --quiet

### Authenticate your notebook environment (Colab only)

If you're running this notebook on Google Colab, run the cell below to authenticate your environment.

In [ ]:
# import sys
#
# if "google.colab" in sys.modules:
#     from google.colab import auth
#
#     auth.authenticate_user()

### Set Google Cloud project information

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [ ]:
# Use the environment variable if the user doesn't provide Project ID.
import logging
import os

import google_auth_oauthlib
import vertexai
from google.genai import types
from vertexai import agent_engines

from dotenv import load_dotenv

from agent import root_agent

load_dotenv()  #

logging.getLogger().setLevel(logging.INFO)

PROJECT_ID = os.environ.get(
    "PROJECT_ID"
)  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "us-central1")

BUCKET_NAME = f"{PROJECT_ID}-bucket"  # @param {type: "string", placeholder: "[your-bucket-name]", isTemplate: true}
if not BUCKET_NAME or BUCKET_NAME == "[your-bucket-name]":
    BUCKET_NAME = PROJECT_ID

BUCKET_URI = f"gs://{BUCKET_NAME}"

# !gsutil mb -l $LOCATION -p $PROJECT_ID $BUCKET_URI

# Initialize Vertex AI session
vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)
# genai.configure(project=PROJECT_ID)

# Initialize the Gen AI client using http_options
# The parameter customizes how the Vertex AI client communicates with Google Cloud's backend services.
# It's used here to access new, pre-release features.
client = vertexai.Client(
    project=PROJECT_ID,
    location=LOCATION,
    http_options=types.HttpOptions(
        api_version="v1beta1", base_url=f"https://{LOCATION}-aiplatform.googleapis.com/"
    ),
)

In [ ]:
PROJECT_NUMBER = os.environ.get("PROJECT_NUMBER")
print(f"PROJECT_NUMBER: {PROJECT_NUMBER}")

### Import libraries

Here, we're importing all the necessary Python classes and functions we'll use throughout the notebook.

In [ ]:
# Logging configured above in imports cell

## ADK root agent

In [ ]:
# TODO change `ENVIRONMENT=production` in `.env`
# root_agent imported above in imports cell

In [ ]:
# Wrap the agent in an AdkApp object
app = agent_engines.AdkApp(
    agent=root_agent,
    enable_tracing=True,
)

In [ ]:
os.getenv("AUTH_ID")

## Deploy to Agent Engine

In [ ]:
remote_agent = agent_engines.create(
    agent_engine=app,
    display_name="ADK_MCP_OAuth_Agent",
    requirements=[
        "google-cloud-aiplatform[agent_engines]>=1.132.0",
        "google-adk>=1.18",
        "fastapi",
        "python-dotenv",
    ],
    extra_packages=["agent.py"],  # Include agent.py in deployment package
    env_vars={
        "MCP_URL": os.getenv("MCP_URL"),
        "AUTH_ID": os.getenv("AUTH_ID", "mcp-oauth0002"),
        "ENVIRONMENT": "production",
        "DEBUG_CONTEXT": "true",  # Enable debug logging
        # Note: GOOGLE_CLIENT_ID and GOOGLE_CLIENT_SECRET are NOT needed in production
        # In production, AgentSpace provides the OAuth token through the context
    },  # Pass environment variables to the deployed agent
)

In [ ]:
remote_agent.resource_name

## Register Agent to Gemini Enterprise

In [ ]:
# take environment variables from .env.

PROJECT_ID = os.getenv("PROJECT_ID")
LOCATION = os.getenv("LOCATION", "us-central1")
GCS_BUCKET = os.getenv("GCS_BUCKET")
PROJECT_NUMBER = os.getenv("PROJECT_NUMBER")
# REASONING_ENGINE_ID = os.getenv("REASONING_ENGINE")
AS_APP = os.getenv("AS_APP")
AUTH_ID = os.getenv("AUTH_ID")
GOOGLE_CLIENT_ID = os.getenv("GOOGLE_CLIENT_ID")

In [ ]:
REASONING_ENGINE = remote_agent.resource_name

In [ ]:
# google_auth_oauthlib imported above in imports cell

### Get OAuth Authorization URL
Add an OAuth 2.0 client with proper scopes and redirect URL:
https://vertexaisearch.cloud.google.com/oauth-redirect

Example below is based on [Google's OAuth 2.0 client](https://support.google.com/googleapi/answer/6158849):
- From the GCP console, search for "OAuth consent screen"
- Create an OAuth 2.0 client for Web application
- Add the redirect URL specified above
- Click **Create**
- Download the `client_secret.json` and upload it here
- Run the Python script below to obtain the authorization URI

For other 3rd party OAuth clients, follow their documentation to obtain authorization information.

Note: make sure your `client_secret.json` file is in the same directory as this notebook, and has the following schema, at least has client_id and client_secret:
```json
{
  "web": {
    "client_id": "String",
    "project_id": "String",
    "auth_uri": "String (URL)",
    "token_uri": "String (URL)",
    "auth_provider_x509_cert_url": "String (URL)",
    "client_secret": "String",
    "redirect_uris": "Array<String (URL)>"
  }
}
``` 

If you don't have a valid `client_secret.json` file, you can manually input the client ID and client secret.  

client_id = ""  
client_secret = ""

In [ ]:
# Required, call the from_client_secrets_file method to retrieve the client ID from a
# client_secret.json file. The client ID (from that file) and access scopes are required. (You can
# also use the from_client_config method, which passes the client configuration as it originally
# appeared in a client secrets file but doesn't access the file itself.)

# IMPORTANT: For this weather MCP server, we only need basic user info scopes
# The weather API itself is public and doesn't require OAuth
# We're using OAuth to identify the user, not to access their Google data
flow = google_auth_oauthlib.flow.Flow.from_client_secrets_file(
    "client_secret.json",
    scopes=[
        "openid",
        "email",
        "profile",
    ],
)

# Required, indicate where the API server will redirect the user after the user completes
# the authorization flow. The redirect URI is required. The value must exactly
# match one of the authorized redirect URIs for the OAuth 2.0 client, which you
# configured in the API Console. If this value doesn't match an authorized URI,
# you will get a 'redirect_uri_mismatch' error.
#
# IMPORTANT FOR GEMINI ENTERPRISE: This MUST be the Vertex AI Search OAuth redirect URL
# Gemini Enterprise will handle the OAuth flow and pass the token to your MCP server
flow.redirect_uri = "https://vertexaisearch.cloud.google.com/oauth-redirect"

# Generate URL for request to Google's OAuth 2.0 server.
# Use kwargs to set optional request parameters.
authorization_url, state = flow.authorization_url(
    # Recommended, enable offline access so that you can refresh an access token without
    # re-prompting the user for permission. Recommended for web server apps.
    access_type="offline",
    # Optional, enable incremental authorization. Recommended as a best practice.
    include_granted_scopes="true",
    # Optional, if your application knows which user is trying to authenticate, it can use this
    # parameter to provide a hint to the Google Authentication Server.
    login_hint="hint@example.com",
    # Optional, set prompt to 'consent' will prompt the user for consent
    prompt="consent",
)
print("OAUTH_AUTH_URI:")
print(authorization_url)

In [ ]:
authorization_url

In [ ]:
os.environ["OAUTH_AUTH_URI"] = authorization_url

In [ ]:
os.environ["REASONING_ENGINE"] = REASONING_ENGINE

In [ ]:
os.getenv("REASONING_ENGINE")

In [ ]:
os.environ["OAUTH_TOKEN_URI"] = "https://oauth2.googleapis.com/token"

### Set Up Auth Server

In [ ]:
%%bash

curl -X POST \
  -H "Authorization: Bearer $(gcloud auth print-access-token)" \
  -H "Content-Type: application/json" \
  -H "X-Goog-User-Project: ${PROJECT_NUMBER}" \
https://discoveryengine.googleapis.com/v1alpha/projects/${PROJECT_NUMBER}/locations/global/authorizations?authorizationId=${AUTH_ID} \
  -d '{
  "name": "projects/${PROJECT_NUMBER}/locations/global/authorizations/${AUTH_ID}",
  "serverSideOauth2": {
      "clientId": "'"${GOOGLE_CLIENT_ID}"'",
      "clientSecret": "'"${GOOGLE_CLIENT_SECRET}"'",
      "authorizationUri": "'"${OAUTH_AUTH_URI}"'",
      "tokenUri": "'"${OAUTH_TOKEN_URI}"'"
    }
  }'

### Link Agent to Gemini Enterprise

In [ ]:
%%bash

curl -X POST \
  -H "Authorization: Bearer $(gcloud auth print-access-token)" \
  -H "Content-Type: application/json" \
  -H "X-Goog-User-Project: ${PROJECT_NUMBER}" \
https://discoveryengine.googleapis.com/v1alpha/projects/${PROJECT_NUMBER}/locations/global/collections/default_collection/engines/${AS_APP}/assistants/default_assistant/agents \
  -d '{
      "displayName": "'"${DISPLAY_NAME}"'",
      "description": "'"${DESCRIPTION}"'",
      "adk_agent_definition": {
        "tool_settings": {
          "tool_description": "'"${TOOL_DESCRIPTION}"'"
        },
        "provisioned_reasoning_engine": {
          "reasoning_engine":
            "'"${REASONING_ENGINE}"'"
        },
        "authorizations": [
          "projects/'"${PROJECT_NUMBER}"'/locations/global/authorizations/'"${AUTH_ID}"'"
        ]
      },
      
  }'

### List Agents

In [ ]:
%%bash
# export PROJECT_NUMBER=PROJECT_NUMBER
# export AS_APP=AGENTSAPCE_APP_ID
curl -X GET -H "Authorization: Bearer $(gcloud auth print-access-token)" \
-H "Content-Type: application/json" \
-H "X-Goog-User-Project: ${PROJECT_NUMBER}" \
"https://discoveryengine.googleapis.com/v1alpha/projects/${PROJECT_NUMBER}/locations/global/collections/default_collection/engines/${AS_APP}/assistants/default_assistant/agents"

### Delete Agent

In [ ]:
os.environ["AGENT_RESOURCE_NAME"] = (
    "your-agent-resource-name"  # Replace with your agent resource name
)

In [ ]:
%%bash
# export PROJECT_NUMBER=PROJECT_NUMBER
# export AGENT_RESOURCE_NAME=AGENT_RESOURCE_NAME

curl -X DELETE \
  -H "Authorization: Bearer $(gcloud auth print-access-token)" \
  -H "Content-Type: application/json" \
  -H "X-Goog-User-Project: ${PROJECT_NUMBER}" \
https://discoveryengine.googleapis.com/v1alpha/${AGENT_RESOURCE_NAME}